In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
Improved particle-filter baseline for the ROGII wellbore TVT task.

The model keeps the baseline2 particle-filter framing:
  * track a latent structural position, TVT + Z, through measured depth
  * compare observed horizontal-well GR against the paired typewell GR curve
  * weight and resample particles as the hidden TVT interval is traversed

Improvements over the notebook version are deliberately local to that method:
  * higher default particle/seed budget
  * robust preprocessing helpers and per-well diagnostics
  * safer fallbacks around filter failures and missing values
  * standalone CLI/notebook-friendly evaluation and submission generation
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from glob import glob
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except ImportError:  # pragma: no cover - tqdm is available on Kaggle, but keep fallback simple.
    def tqdm(items: Iterable, **_: object) -> Iterable:
        return items


KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
LOCAL_DATA_DIR = Path(".")


@dataclass(frozen=True)
class TrackerConfig:
    n_particles: int = 700
    n_seeds: int = 24
    momentum: float = 0.998
    rate_noise: float = 0.002
    pos_noise: float = 0.005
    resample_threshold: float = 0.50
    likelihood_temp: float = 3.0
    max_tvt_step: float = 2.0
    typewell_margin: float = 180.0
    inject_fraction: float = 0.0
    injection_period: int = 80


@dataclass(frozen=True)
class GRCalibration:
    slope: float
    intercept: float
    sigma: float
    quality: float


@dataclass(frozen=True)
class WellContext:
    hw: pd.DataFrame
    tw_tvt: np.ndarray
    tw_gr: np.ndarray
    tw_gr_smooth: np.ndarray
    hw_gr: np.ndarray
    hw_gr_smooth: np.ndarray
    obs_weight: np.ndarray
    calibration: GRCalibration
    known_mask: np.ndarray
    predict_mask: np.ndarray
    init_pos_rate: float
    init_tvt_rate: float
    trend_step: float


In [ ]:
def auto_data_dir(requested: str | None = None) -> Path:
    if requested:
        return Path(requested)
    if KAGGLE_DATA_DIR.exists():
        return KAGGLE_DATA_DIR
    return LOCAL_DATA_DIR


def default_output_path(data_dir: Path) -> Path:
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working/submission.csv")
    return data_dir / "submission_baseline2_improved.csv"


In [ ]:
def robust_std(values: np.ndarray, default: float = 1.0) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return default
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    if mad > 1e-12:
        return float(1.4826 * mad)
    std = np.std(arr)
    return float(std) if std > 1e-12 else default


def rolling_smooth(values: np.ndarray, median_window: int = 5, mean_window: int = 5) -> np.ndarray:
    s = pd.Series(values, dtype="float64")
    sm = s.rolling(median_window, center=True, min_periods=1).median()
    sm = sm.rolling(mean_window, center=True, min_periods=1).mean()
    return sm.to_numpy(dtype=float)


def fill_and_smooth_gr(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    raw = pd.Series(values, dtype="float64").replace([np.inf, -np.inf], np.nan)
    valid = raw.notna().to_numpy()
    finite = raw.dropna().to_numpy(dtype=float)

    if finite.size:
        lo, hi = np.percentile(finite, [0.5, 99.5])
        raw = raw.clip(lo, hi)
        fill_value = float(np.median(finite))
    else:
        fill_value = 0.0

    filled = raw.interpolate(limit_direction="both").fillna(fill_value).to_numpy(dtype=float)
    smooth = rolling_smooth(filled)
    return filled, smooth, valid


def missing_observation_weights(valid: np.ndarray, n: int) -> np.ndarray:
    if n == 0:
        return np.array([], dtype=float)
    if valid.all():
        return np.ones(n, dtype=float)
    if not valid.any():
        return np.zeros(n, dtype=float)

    idx = np.arange(n)
    prev_valid = np.full(n, -10**9, dtype=int)
    last = -10**9
    for i in range(n):
        if valid[i]:
            last = i
        prev_valid[i] = last

    next_valid = np.full(n, 10**9, dtype=int)
    nxt = 10**9
    for i in range(n - 1, -1, -1):
        if valid[i]:
            nxt = i
        next_valid[i] = nxt

    dist = np.minimum(idx - prev_valid, next_valid - idx).astype(float)
    weights = np.where(valid, 1.0, 0.30 * np.exp(-dist / 25.0))
    return np.clip(weights, 0.0, 1.0)


In [ ]:
def prepare_typewell(tw: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    tw_s = tw[["TVT", "GR"]].copy()
    tw_s = tw_s.replace([np.inf, -np.inf], np.nan).dropna(subset=["TVT"])
    tw_s = tw_s.groupby("TVT", as_index=False)["GR"].mean().sort_values("TVT")
    tw_s["GR"] = tw_s["GR"].interpolate(limit_direction="both")
    if tw_s["GR"].isna().any():
        tw_s["GR"] = tw_s["GR"].fillna(float(tw_s["GR"].median()))

    tw_tvt = tw_s["TVT"].to_numpy(dtype=float)
    tw_gr = tw_s["GR"].to_numpy(dtype=float)
    tw_gr_smooth = rolling_smooth(tw_gr)
    return tw_tvt, tw_gr, tw_gr_smooth


In [ ]:
def fit_gr_calibration(
    hw: pd.DataFrame,
    tw_tvt: np.ndarray,
    tw_gr_smooth: np.ndarray,
    hw_gr_smooth: np.ndarray,
    observed_gr_mask: np.ndarray,
) -> GRCalibration:
    known_mask = hw["TVT_input"].notna().to_numpy()
    fit_mask = known_mask & observed_gr_mask

    if fit_mask.sum() < 12:
        observed = hw_gr_smooth[observed_gr_mask]
        intercept = float(np.median(observed) - np.median(tw_gr_smooth)) if observed.size else 0.0
        return GRCalibration(1.0, intercept, 28.0, 0.0)

    x = np.interp(hw.loc[fit_mask, "TVT_input"].to_numpy(dtype=float), tw_tvt, tw_gr_smooth)
    y = hw_gr_smooth[fit_mask]
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if x.size < 12 or np.std(x) < 1e-6:
        intercept = float(np.median(y - x)) if x.size else 0.0
        resid = y - (x + intercept) if x.size else np.array([28.0])
        return GRCalibration(1.0, intercept, float(np.clip(robust_std(resid, 28.0), 8.0, 60.0)), 0.1)

    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    keep_scale = robust_std(resid, 20.0)
    keep = np.abs(resid - np.median(resid)) <= 3.5 * max(keep_scale, 1.0)
    if keep.sum() >= 12 and keep.sum() < x.size:
        slope, intercept = np.polyfit(x[keep], y[keep], 1)
        resid = y[keep] - (slope * x[keep] + intercept)

    slope = float(np.clip(slope, 0.35, 2.25))
    intercept = float(np.median(y - slope * x))
    resid = y - (slope * x + intercept)
    sigma = float(np.clip(robust_std(resid, 20.0), 7.5, 60.0))

    corr = np.corrcoef(x, y)[0, 1] if x.size >= 3 and np.std(y) > 1e-6 else 0.0
    quality = float(np.clip(abs(corr), 0.0, 1.0))
    return GRCalibration(slope, intercept, sigma, quality)


In [ ]:
def estimate_initial_rates(hw: pd.DataFrame, known_mask: np.ndarray) -> tuple[float, float, float]:
    known = hw.loc[known_mask, ["MD", "Z", "TVT_input"]].dropna()
    if len(known) < 3:
        return 0.0, 0.0, 0.05

    tail = known.tail(min(30, len(known)))
    md = tail["MD"].to_numpy(dtype=float)
    tvt = tail["TVT_input"].to_numpy(dtype=float)
    z = tail["Z"].to_numpy(dtype=float)
    dmd = np.diff(md)
    ok = dmd > 0
    if ok.sum() < 2:
        return 0.0, 0.0, 0.05

    pos_slope = np.diff(tvt + z)[ok] / dmd[ok]
    tvt_slope = np.diff(tvt)[ok] / dmd[ok]
    pos_slope = pos_slope[np.isfinite(pos_slope)]
    tvt_slope = tvt_slope[np.isfinite(tvt_slope)]

    init_pos_rate = float(np.clip(np.median(pos_slope), -0.12, 0.12)) if pos_slope.size else 0.0
    init_tvt_rate = float(np.clip(np.median(tvt_slope), -0.12, 0.12)) if tvt_slope.size else 0.0

    tvt_step = np.abs(np.diff(tvt))
    tvt_step = tvt_step[np.isfinite(tvt_step)]
    trend_step = float(np.percentile(tvt_step, 99)) if tvt_step.size else 0.05
    trend_step = float(np.clip(max(trend_step, 0.05), 0.05, 1.5))
    return init_pos_rate, init_tvt_rate, trend_step


In [ ]:
def build_context(hw: pd.DataFrame, tw: pd.DataFrame) -> WellContext:
    hw = hw.copy()
    tw_tvt, tw_gr, tw_gr_smooth = prepare_typewell(tw)
    hw_gr, hw_gr_smooth, observed_mask = fill_and_smooth_gr(hw["GR"].to_numpy(dtype=float))
    obs_weight = missing_observation_weights(observed_mask, len(hw))
    calibration = fit_gr_calibration(hw, tw_tvt, tw_gr_smooth, hw_gr_smooth, observed_mask)
    known_mask = hw["TVT_input"].notna().to_numpy()
    predict_mask = ~known_mask
    init_pos_rate, init_tvt_rate, trend_step = estimate_initial_rates(hw, known_mask)

    return WellContext(
        hw=hw,
        tw_tvt=tw_tvt,
        tw_gr=tw_gr,
        tw_gr_smooth=tw_gr_smooth,
        hw_gr=hw_gr,
        hw_gr_smooth=hw_gr_smooth,
        obs_weight=obs_weight,
        calibration=calibration,
        known_mask=known_mask,
        predict_mask=predict_mask,
        init_pos_rate=init_pos_rate,
        init_tvt_rate=init_tvt_rate,
        trend_step=trend_step,
    )


In [ ]:
def calibrated_typewell_gr(tvt: np.ndarray, ctx: WellContext) -> np.ndarray:
    tw_lookup = np.interp(tvt, ctx.tw_tvt, ctx.tw_gr_smooth)
    return ctx.calibration.slope * tw_lookup + ctx.calibration.intercept


def systematic_resample(weights: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    n = weights.size
    positions = (rng.random() + np.arange(n)) / n
    cumulative = np.cumsum(weights)
    return np.clip(np.searchsorted(cumulative, positions), 0, n - 1)


def score_path(path: np.ndarray, ctx: WellContext) -> float:
    idx = np.where(ctx.predict_mask)[0]
    if idx.size == 0:
        return -1e9
    weights = ctx.obs_weight[idx]
    if weights.sum() <= 1e-9:
        return 0.0

    expected = calibrated_typewell_gr(path[idx], ctx)
    resid = (ctx.hw_gr_smooth[idx] - expected) / max(ctx.calibration.sigma, 1.0)
    clipped = np.minimum(resid * resid, 80.0)
    return float(-0.5 * np.sum(weights * clipped))


def clamp_prediction_step(candidate: float, previous: float, ctx: WellContext, cfg: TrackerConfig) -> float:
    limit = max(cfg.max_tvt_step, 6.0 * ctx.trend_step)
    if abs(candidate - previous) <= limit:
        return candidate
    return previous + np.sign(candidate - previous) * limit


In [ ]:
def run_particle_filter(ctx: WellContext, cfg: TrackerConfig, seed: int) -> tuple[np.ndarray, float]:
    hw = ctx.hw
    md_arr = hw["MD"].to_numpy(dtype=float)
    z_arr = hw["Z"].to_numpy(dtype=float)
    tvt_input = hw["TVT_input"].to_numpy(dtype=float)
    known_idx = np.where(ctx.known_mask)[0]
    pred_idx = np.where(ctx.predict_mask)[0]
    out = tvt_input.copy()

    if pred_idx.size == 0 or known_idx.size == 0:
        return np.nan_to_num(out, nan=0.0), 0.0

    last_known = known_idx[-1]
    last_tvt = float(tvt_input[last_known])
    last_z = float(z_arr[last_known])
    last_md = float(md_arr[last_known])
    last_pos = last_tvt + last_z

    rng = np.random.default_rng(seed)
    init_spread = 2.0
    rate_spread = 0.010

    pos = last_pos + init_spread * rng.standard_normal(cfg.n_particles)
    rate = ctx.init_pos_rate + rate_spread * rng.standard_normal(cfg.n_particles)
    weights = np.ones(cfg.n_particles, dtype=float) / cfg.n_particles

    prev_md = last_md
    prev_tvt = last_tvt
    lower = float(ctx.tw_tvt.min() - 100.0)
    upper = float(ctx.tw_tvt.max() + 100.0)
    log_lik = 0.0

    pf_gr = (
        hw["GR"]
        .interpolate(limit_direction="both")
        .fillna(float(np.nanmean(ctx.tw_gr)))
        .to_numpy(dtype=float)
    )
    known_expected = np.interp(tvt_input[ctx.known_mask], ctx.tw_tvt, ctx.tw_gr)
    known_gr = hw.loc[ctx.known_mask, "GR"].fillna(0.0).to_numpy(dtype=float)
    gr_sigma = float(np.clip(np.nanstd(known_gr - known_expected), 10.0, 60.0))

    for step, i in enumerate(pred_idx):
        md_i = float(md_arr[i])
        z_i = float(z_arr[i])
        dm = max(md_i - prev_md, 1.0)

        rate = cfg.momentum * rate + cfg.rate_noise * rng.standard_normal(cfg.n_particles)
        pos = pos + rate * dm + cfg.pos_noise * rng.standard_normal(cfg.n_particles)

        tvt_particles = np.clip(pos - z_i, lower, upper)
        pos = tvt_particles + z_i

        expected = np.interp(tvt_particles, ctx.tw_tvt, ctx.tw_gr)
        resid = (pf_gr[i] - expected) / gr_sigma
        likelihood = np.maximum(np.exp(-0.5 * np.minimum(resid * resid, 600.0)), 1e-300)
        avg_likelihood = float(np.sum(weights * likelihood))
        log_lik += np.log(max(avg_likelihood, 1e-300))
        weights *= likelihood
        total = weights.sum()
        if total > 0 and np.isfinite(total):
            weights /= total
        else:
            weights.fill(1.0 / cfg.n_particles)

        n_eff = 1.0 / np.sum(weights * weights)
        should_resample = n_eff < cfg.resample_threshold * cfg.n_particles
        if should_resample:
            picked = systematic_resample(weights, rng)
            pos = pos[picked] + 0.1 * rng.standard_normal(cfg.n_particles)
            rate = rate[picked] + 0.001 * rng.standard_normal(cfg.n_particles)
            weights.fill(1.0 / cfg.n_particles)

        if cfg.inject_fraction > 0 and (should_resample or (step + 1) % cfg.injection_period == 0):
            n_inject = max(1, int(cfg.inject_fraction * cfg.n_particles))
            inject_idx = rng.choice(cfg.n_particles, size=n_inject, replace=False)
            trend_tvt = last_tvt + ctx.init_tvt_rate * (md_i - last_md)
            trend_pos = trend_tvt + z_i
            pos[inject_idx] = trend_pos + rng.normal(0.0, max(2.0, 8.0 * ctx.trend_step), size=n_inject)
            rate[inject_idx] = ctx.init_pos_rate + rng.normal(0.0, 0.006, size=n_inject)

        tvt_est = float(np.dot(weights, pos - z_i))
        tvt_est = clamp_prediction_step(tvt_est, prev_tvt, ctx, cfg)
        out[i] = tvt_est
        prev_tvt = tvt_est
        prev_md = md_i

    return np.clip(np.nan_to_num(out, nan=last_tvt), lower, upper), float(log_lik)


In [ ]:
def trend_path(ctx: WellContext, cfg: TrackerConfig, mode: str) -> np.ndarray:
    hw = ctx.hw
    md_arr = hw["MD"].to_numpy(dtype=float)
    z_arr = hw["Z"].to_numpy(dtype=float)
    tvt_input = hw["TVT_input"].to_numpy(dtype=float)
    out = tvt_input.copy()
    known_idx = np.where(ctx.known_mask)[0]
    pred_idx = np.where(ctx.predict_mask)[0]
    if known_idx.size == 0:
        return np.nan_to_num(out, nan=float(np.median(ctx.tw_tvt)))

    last = known_idx[-1]
    last_tvt = float(tvt_input[last])
    last_z = float(z_arr[last])
    last_md = float(md_arr[last])
    last_pos = last_tvt + last_z
    prev = last_tvt
    lower = float(ctx.tw_tvt.min() - cfg.typewell_margin)
    upper = float(ctx.tw_tvt.max() + cfg.typewell_margin)

    for i in pred_idx:
        md_i = float(md_arr[i])
        z_i = float(z_arr[i])
        if mode == "constant":
            candidate = last_tvt
        elif mode == "tvt_linear":
            candidate = last_tvt + ctx.init_tvt_rate * (md_i - last_md)
        else:
            candidate = last_pos + ctx.init_pos_rate * (md_i - last_md) - z_i
        candidate = float(np.clip(candidate, lower, upper))
        candidate = clamp_prediction_step(candidate, prev, ctx, cfg)
        out[i] = candidate
        prev = candidate

    return np.clip(np.nan_to_num(out, nan=last_tvt), lower, upper)


def local_gr_search_path(ctx: WellContext, cfg: TrackerConfig) -> np.ndarray:
    hw = ctx.hw
    md_arr = hw["MD"].to_numpy(dtype=float)
    tvt_input = hw["TVT_input"].to_numpy(dtype=float)
    out = tvt_input.copy()
    known_idx = np.where(ctx.known_mask)[0]
    pred_idx = np.where(ctx.predict_mask)[0]
    if known_idx.size == 0:
        return np.nan_to_num(out, nan=float(np.median(ctx.tw_tvt)))

    last = known_idx[-1]
    last_tvt = float(tvt_input[last])
    last_md = float(md_arr[last])
    prev = last_tvt
    lower = float(ctx.tw_tvt.min() - cfg.typewell_margin)
    upper = float(ctx.tw_tvt.max() + cfg.typewell_margin)

    for step, i in enumerate(pred_idx):
        md_i = float(md_arr[i])
        prev_md = float(md_arr[pred_idx[step - 1]]) if step else last_md
        center = prev + ctx.init_tvt_rate * max(md_i - prev_md, 0.25)
        distance_from_anchor = md_i - last_md
        radius = float(np.clip(4.0 + 0.0015 * distance_from_anchor, 4.0, 16.0))
        candidates = np.linspace(center - radius, center + radius, 161)
        candidates = np.clip(candidates, lower, upper)

        obs_w = float(ctx.obs_weight[i])
        prior = ((candidates - center) / max(radius * 0.55, 1.0)) ** 2
        if obs_w > 1e-6:
            expected = calibrated_typewell_gr(candidates, ctx)
            resid = (ctx.hw_gr_smooth[i] - expected) / max(ctx.calibration.sigma, 1.0)
            score = obs_w * resid * resid + 0.35 * prior
        else:
            score = prior

        chosen = float(candidates[int(np.argmin(score))])
        chosen = clamp_prediction_step(chosen, prev, ctx, cfg)
        out[i] = chosen
        prev = chosen

    return np.clip(np.nan_to_num(out, nan=last_tvt), lower, upper)


In [ ]:
def predict_well(wid: str, hw: pd.DataFrame, tw: pd.DataFrame, cfg: TrackerConfig) -> tuple[np.ndarray, dict[str, float]]:
    ctx = build_context(hw, tw)
    if ctx.predict_mask.sum() == 0:
        pred = ctx.hw["TVT_input"].to_numpy(dtype=float)
        return pred, {"best_score": 0.0, "cal_sigma": ctx.calibration.sigma, "cal_quality": ctx.calibration.quality}

    candidates: list[np.ndarray] = []
    scores: list[float] = []

    for seed in range(cfg.n_seeds):
        try:
            path, log_lik = run_particle_filter(ctx, cfg, seed=seed)
        except Exception:
            path = trend_path(ctx, cfg, mode="pos_linear")
            log_lik = -1e12
        candidates.append(path)
        scores.append(log_lik)

    scores = np.array(scores, dtype=float)
    scores = np.where(np.isfinite(scores), scores, -1e12)
    shifted = (scores - scores.max()) / max(cfg.likelihood_temp, 1e-6)
    weights = np.exp(np.clip(shifted, -80.0, 0.0))
    weight_sum = weights.sum()
    if weight_sum <= 0:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights /= weight_sum

    stacked = np.stack(candidates, axis=0)
    pred = np.sum(weights[:, None] * stacked, axis=0)
    known_values = ctx.hw["TVT_input"].to_numpy(dtype=float)
    pred[ctx.known_mask] = known_values[ctx.known_mask]

    lower = float(ctx.tw_tvt.min() - cfg.typewell_margin)
    upper = float(ctx.tw_tvt.max() + cfg.typewell_margin)
    pred = np.clip(np.nan_to_num(pred, nan=float(np.nanmedian(known_values))), lower, upper)

    diagnostics = {
        "best_score": float(scores.max()),
        "cal_sigma": float(ctx.calibration.sigma),
        "cal_quality": float(ctx.calibration.quality),
        "best_candidate": float(np.argmax(scores)),
        "best_weight": float(weights.max()),
    }
    return pred, diagnostics


In [ ]:
def evaluate(data_dir: Path, cfg: TrackerConfig, n_wells: int | None = 20, seed: int = 42) -> None:
    train_dir = data_dir / "train"
    files = sorted(train_dir.glob("*__horizontal_well.csv"))
    if n_wells is not None and n_wells > 0 and n_wells < len(files):
        rng = np.random.default_rng(seed)
        files = [files[i] for i in rng.choice(len(files), size=n_wells, replace=False)]

    rmses: list[float] = []
    maes: list[float] = []
    all_errs: list[np.ndarray] = []

    for hw_path in tqdm(files, desc="[Eval] Wells"):
        wid = hw_path.name.replace("__horizontal_well.csv", "")
        tw_path = train_dir / f"{wid}__typewell.csv"
        if not tw_path.exists():
            continue
        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path)
        mask = hw["TVT_input"].isna().to_numpy()
        if mask.sum() == 0 or "TVT" not in hw.columns:
            continue

        pred, diag = predict_well(wid, hw, tw, cfg)
        true = hw.loc[mask, "TVT"].to_numpy(dtype=float)
        err = pred[mask] - true
        if not np.isfinite(err).all():
            continue
        rmse = float(np.sqrt(np.mean(err * err)))
        mae = float(np.mean(np.abs(err)))
        rmses.append(rmse)
        maes.append(mae)
        all_errs.append(err)
        print(
            f"{wid}: RMSE={rmse:.3f} MAE={mae:.3f} "
            f"sigma={diag['cal_sigma']:.1f} q={diag['cal_quality']:.2f}"
        )

    if not rmses:
        print("No evaluable wells found.")
        return

    pooled = np.concatenate(all_errs)
    print("\n=== Evaluation ===")
    print(f"Wells: {len(rmses)}")
    print(f"Per-well RMSE mean/median/worst: {np.mean(rmses):.3f} / {np.median(rmses):.3f} / {np.max(rmses):.3f}")
    print(f"Per-well MAE  mean/median:       {np.mean(maes):.3f} / {np.median(maes):.3f}")
    print(f"Pooled RMSE/MAE/bias:            {np.sqrt(np.mean(pooled * pooled)):.3f} / {np.mean(np.abs(pooled)):.3f} / {np.mean(pooled):.3f}")


In [ ]:
def generate_submission(data_dir: Path, output_path: Path, cfg: TrackerConfig) -> pd.DataFrame:
    test_dir = data_dir / "test"
    sample_path = data_dir / "sample_submission.csv"
    if not sample_path.exists():
        raise FileNotFoundError(f"Missing sample submission: {sample_path}")

    sub = pd.read_csv(sample_path)
    predictions: dict[str, np.ndarray] = {}

    for hw_file in tqdm(sorted(glob(str(test_dir / "*__horizontal_well.csv"))), desc="[Predict] Wells"):
        hw_path = Path(hw_file)
        wid = hw_path.name.replace("__horizontal_well.csv", "")
        tw_path = test_dir / f"{wid}__typewell.csv"
        if not tw_path.exists():
            raise FileNotFoundError(f"Missing typewell for {wid}: {tw_path}")

        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path)
        pred, diag = predict_well(wid, hw, tw, cfg)
        predictions[wid] = pred
        hidden = int(hw["TVT_input"].isna().sum())
        print(
            f"{wid}: rows={len(hw)} hidden={hidden} "
            f"TVT=[{np.nanmin(pred):.2f}, {np.nanmax(pred):.2f}] "
            f"sigma={diag['cal_sigma']:.1f} q={diag['cal_quality']:.2f}"
        )

    ids = sub["id"].astype(str)
    for idx, item_id in enumerate(ids):
        wid, row_idx = item_id.rsplit("_", 1)
        row_idx_i = int(row_idx)
        if wid in predictions and row_idx_i < len(predictions[wid]):
            sub.at[idx, "tvt"] = float(predictions[wid][row_idx_i])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(output_path, index=False)
    print(f"Submission saved: {output_path} ({len(sub)} rows)")
    return sub


In [ ]:
# Set these variables before running the final cell.
DATA_DIR = None          # None auto-detects Kaggle input or local current directory.
OUTPUT_PATH = None       # None writes /kaggle/working/submission.csv on Kaggle, else local submission_baseline2_improved.csv.

RUN_EVAL = False         # Set True to run local train-prefix evaluation.
EVAL_WELLS = 20          # Use <= 0 to evaluate all train wells.
EVAL_SEED = 42

RUN_SUBMISSION = True    # Set False if you only want evaluation.
N_PARTICLES = 700
N_SEEDS = 24


In [ ]:
data_dir = auto_data_dir(DATA_DIR)
output_path = Path(OUTPUT_PATH) if OUTPUT_PATH else default_output_path(data_dir)
cfg = TrackerConfig(n_particles=N_PARTICLES, n_seeds=N_SEEDS)

In [ ]:
output_path

In [ ]:
if RUN_EVAL:
    n_wells = None if EVAL_WELLS <= 0 else EVAL_WELLS
    evaluate(data_dir, cfg, n_wells=n_wells, seed=EVAL_SEED)

if RUN_SUBMISSION:
    generate_submission(data_dir, output_path, cfg)
